In [1]:
# %%
import datetime
import json
import multiprocessing as mp
import os
import pathlib
import re
import sys
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from functools import partial
from dateutil.relativedelta import relativedelta

# import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tqdm
import xarray as xr

2025-06-03 11:44:57.823922: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-03 11:44:57.824035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-03 11:44:57.835632: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
src_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs')
boundary_path = pathlib.Path(
    "/gpfs/work2/0/prjs1027/guus/data/BoundaryConditions/bnd_storms.nc"
)
write_dir = pathlib.Path("/gpfs/work2/0/prjs1027/tfrecords/run2")
write_dir.mkdir(exist_ok=True, parents=True)
write_dir = pathlib.Path("/gpfs/work2/0/prjs1027/guus/data")
filtered_paths = [
    path for path in src_path.glob("ens*") if re.search(r"\d+$", str(path))
]
filtered_paths = [path for path in src_path.glob('0*') if path.stem[-4:] == '2018']
# ens = os.environ.get("ENS")
ensembles = [26, 27, 28, 29, 30]

In [3]:
boundary_ds = xr.open_dataset(boundary_path)

# %%
output_variables = ["hs", "theta0_x", "theta0_y", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "ssh", "xcur", "ycur"]
boundary_variables = ["hs", "hswe", "tmm10"]
conversion_dict = {"hs": "Hm0", "hswe": "HE10", "tmm10": "Tmm10"}

In [4]:
def gather_input(path, boundary_ds, variables, boundary_variables):
    datasets = []
    input_fields = {var: [] for var in variables}
    boundary_fields = {var: [] for var in boundary_variables}
    for _ in tqdm.tqdm(list(path.glob("*"))):
        year = _.stem[:4]
        month = _.stem[4:6]
        year_month = f'{year}-{month}'
        start_date = datetime.datetime.strptime(year_month, '%Y-%m')
        if year == '2013':
            end_date = (start_date + relativedelta(days=6)) - relativedelta(minutes=1)
        else:
            end_date = (start_date + relativedelta(months=1)) - relativedelta(minutes=1)
        try:
            ds_path = next(_.glob("output/*.nc"))
            ds = xr.open_dataset(ds_path)
            ds = ds.sel(time=slice(start_date, end_date))
            ds = ds.resample(time='6H').nearest()
        except:
            # print(f"{_ / 'output/swan2d.nc'} does not exist, skipping")
            continue

        datasets.append(ds)

    print(f"Done with {path}")
    return datasets

In [5]:
# Define the function to process each path
def process_path(paths, ens):
    datasets = []
    for path in paths:
        ens_path = path / f'computations/ens{ens}'
        print(f'Processing path {ens_path}')
        dataset = gather_input(
            ens_path,
            boundary_ds=boundary_ds,
            variables=input_variables + output_variables,
            boundary_variables=boundary_variables,
        )
        datasets.append(xr.concat([ds for ds in dataset], dim='time'))
        print(f"Processed {ens_path}")
    return datasets

In [9]:
for ens in ensembles:
    datasets = process_path(filtered_paths, ens)

    ds = xr.concat(datasets, dim='time')
    ds = ds.sortby('time')

    ds = ds.drop_duplicates(dim="time", keep="first")
    ds.to_netcdf(write_dir / f'storm_2018_ens{ens}.nc')

Processing path /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens26


100%|██████████| 30/30 [00:00<00:00, 36.99it/s]


Done with /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens26
Processed /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens26
Processing path /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens27


100%|██████████| 30/30 [00:00<00:00, 33.04it/s]


Done with /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens27
Processed /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens27
Processing path /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens28


...

Done with /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens28
Processed /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens28
Processing path /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens29


100%|██████████| 30/30 [00:00<00:00, 34.74it/s]


Done with /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens29
Processed /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens29
Processing path /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens30


100%|██████████| 30/30 [00:01<00:00, 28.20it/s]


Done with /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens30
Processed /gpfs/work2/0/prjs1027/swn-runs/06-jan2018/computations/ens30


In [7]:
ds = xr.concat(datasets, dim='time')
ds = ds.sortby('time')

In [8]:
ds = ds.drop_duplicates(dim="time", keep="first")
ds.to_netcdf(write_dir / 'training_storm_ens10.nc')

In [11]:
datasets = list(src_path.glob('*/output/*.nc'))

def preprocess(ds):
    return ds.sortby('time')

In [12]:
datasets.sort()

In [13]:
ds = xr.concat([xr.open_dataset(data) for data in datasets], dim='time')
ds = ds.drop_duplicates(dim="time", keep="first")
ds.to_netcdf(write_dir / 'training_storm2018_ens10.nc')

In [14]:
ds = xr.load_dataset(write_dir / 'training_storm2018_ens10.nc')

In [15]:
ds

<xarray.Dataset>
Dimensions:    (latitude: 481, longitude: 421, time: 125)
Coordinates:
  * latitude   (latitude) float32 48.0 48.03 48.07 48.1 ... 63.93 63.97 64.0
  * longitude  (longitude) float32 -12.0 -11.95 -11.9 -11.85 ... 8.9 8.95 9.0
  * time       (time) datetime64[ns] 2018-01-01 ... 2018-02-01
Data variables:
    hs         (time, latitude, longitude) float32 7.434 7.437 ... 0.007629
    hswe       (time, latitude, longitude) float32 6.256 6.261 ... 0.006104
    tmm10      (time, latitude, longitude) float32 11.98 11.99 ... 9.968 10.02
    tps        (time, latitude, longitude) float32 14.22 14.22 ... 11.46 11.47
    theta0     (time, latitude, longitude) float32 268.0 268.0 ... 256.0 256.9
    fdir       (time, latitude, longitude) float32 270.7 270.7 ... 257.2 258.0
    spread     (time, latitude, longitude) float32 29.04 29.0 ... 9.69 8.685
    ssh        (time, latitude, longitude) float32 0.358 0.3589 ... 0.0 0.0
    xcur       (time, latitude, longitude) float32 0.04456 0.04456 ... nan nan
    ycur       (time, latitude, longitude) float32 -0.006714 -0.006714 ... nan
    xwnd       (time, latitude, longitude) float32 nan nan nan ... nan nan nan
    ywnd       (time, latitude, longitude) float32 nan nan nan ... nan nan nan
Attributes:
    Conventions:             CF-1.5
    History:                 Created with agioncmd version 1.5
    Directional_convention:  nautical
    project:                 SWAN-EPS
    run:                     030

In [13]:
ds.get_index('longitude').get_loc(3.25)
ds.get_index('latitude').get_loc(52.0)

120

In [12]:
ds.sel(latitude=51.998611, longitude=3.276389, method="nearest")

<xarray.Dataset>
Dimensions:    (time: 1457)
Coordinates:
    latitude   float32 52.0
    longitude  float32 3.3
  * time       (time) datetime64[ns] 2022-01-01 ... 2022-12-31
Data variables:
    hs         (time) float32 0.3616 0.959 1.323 1.359 ... 1.885 2.064 1.652
    hswe       (time) float32 0.0 0.0007629 0.003815 ... 0.0885 0.09537 0.1038
    tmm10      (time) float32 2.361 4.216 4.182 5.0 ... 4.989 5.191 5.249 5.472
    tps        (time) float32 2.152 4.321 5.225 5.325 ... 5.707 6.294 6.362
    theta0     (time) float32 38.01 225.5 224.7 229.4 ... 196.5 223.4 232.8
    fdir       (time) float32 nan 238.9 232.1 256.6 ... 335.5 328.5 313.4 324.9
    spread     (time) float32 31.51 31.04 29.58 28.57 ... 37.48 27.41 29.09
    ssh        (time) float32 0.9856 -0.6221 0.8071 ... -1.133 0.6111 -0.3603
    xcur       (time) float32 0.4077 -0.3638 0.4669 ... -0.1489 0.4364 0.04273
    ycur       (time) float32 0.556 -0.5365 0.5963 ... -0.08606 0.3803 0.1813
    xwnd       (time) float32 7.114 4.26 5.206 5.857 ... 3.159 -3.33 13.68 5.197
    ywnd       (time) float32 5.56 7.987 10.16 8.283 ... 11.59 14.27 5.533 7.608
Attributes:
    Conventions:             CF-1.5
    History:                 Created with agioncmd version 1.5
    Directional_convention:  nautical
    project:                 SWAN-EPS
    run:                     030

In [11]:
ds.drop_duplicates(dim="time", keep="first")

<xarray.Dataset>
Dimensions:    (time: 739, longitude: 421, latitude: 481)
Coordinates:
  * time       (time) datetime64[ns] 2013-12-01 ... 2023-12-31
  * longitude  (longitude) float32 -12.0 -11.95 -11.9 -11.85 ... 8.9 8.95 9.0
  * latitude   (latitude) float32 48.0 48.03 48.07 48.1 ... 63.93 63.97 64.0
Data variables:
    hs         (time, latitude, longitude) float32 2.242 2.242 ... 0.3327 0.2197
    hswe       (time, latitude, longitude) float32 1.171 1.171 ... 0.2152 0.2014
    tmm10      (time, latitude, longitude) float32 7.917 7.916 ... 6.846 11.73
    tps        (time, latitude, longitude) float32 10.3 10.3 ... 12.73 12.76
    theta0     (time, latitude, longitude) float32 358.2 358.1 ... 196.7 256.2
    fdir       (time, latitude, longitude) float32 327.8 327.6 ... 257.5 258.4
    spread     (time, latitude, longitude) float32 48.04 48.02 ... 59.43 14.73
    ssh        (time, latitude, longitude) float32 0.4326 0.4349 ... 0.0 0.0
    xcur       (time, latitude, longitude) float32 0.03113 0.03052 ... nan nan
    ycur       (time, latitude, longitude) float32 0.01038 0.01221 ... nan nan
    xwnd       (time, latitude, longitude) float32 -8.502 -8.478 ... -6.128
    ywnd       (time, latitude, longitude) float32 -3.214 -3.244 ... 2.731 2.829
Attributes:
    Conventions:             CF-1.5
    History:                 Created with agioncmd version 1.5
    Directional_convention:  nautical
    project:                 SWAN-EPS
    run:                     030